### Data Ingestion (Bronze)

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *
from datetime import date, timedelta

yesterday = (date.today() - timedelta(days=1)).strftime('%Y-%m-%d')
path = f's3://ruturaj-serverless/bank_project/bronze/transactions/{yesterday}/transactions.csv'

new_schema = StructType([

    StructField('transaction_id', StringType()),
    StructField('customer_id', IntegerType()),
    StructField('amount', DecimalType(10,2)),
    StructField('transaction_date', DateType()),
    StructField('status', StringType()),
    StructField('description', StringType()),
    StructField('merchant_id', IntegerType()),
    StructField('payment_type', StringType()),
    StructField('currency', StringType()),
    StructField('location', StringType()),
    StructField('notes', StringType()),

])

df = spark.read.schema(new_schema).csv(path, header=True)

transactions = df

### Data Exploration

In [0]:
def check_null_count(df):

  if type(df).__name__ != 'DataFrame':
    raise TypeError(f'Expected DataFrme as parameter. Given {type(df).__name__}.')

  header = ['column', 'null_count', 'data_Type']

  null_count = df.select([sum(col(c).isNull().cast('int')).alias(c) for c in df.columns]).collect()[0]

  columns = [c[0] for c in df.dtypes]
  null_count = [c for c in null_count]
  column_dtype = [c[1] for c in df.dtypes]

  rows = [(c, n, d) for c, n, d in zip(columns, null_count, column_dtype)]

  return spark.createDataFrame(rows, header)

transactions_null_check = check_null_count(transactions)
transactions_null_check.display()


### Data Cleaaning

In [0]:
def clean_string_data(df):

    if type(df).__name__ != 'DataFrame':
        raise TypeError(f'Expected DataFrme as parameter. Given {type(df).__name__}.')

    expr = []
    for c, d in df.dtypes:

        if d == 'string':
            expr.append(upper(trim(col(c))).alias(c))
        else:
            expr.append(col(c))

    return df.select(expr)

transactions = transactions.dropna(subset=['transaction_id', 'customer_id'])

transactions = transactions.fillna(
                    {
                        'amount': 0,
                        'status': 'pending',
                        'description':'',
                        'payment_type': 'bank_transfer',
                        'currency': 'inr',
                        'location': '',
                        'notes': ''
                    }
)

transactions = clean_string_data(transactions)

# transactions.display()


### Data Loading (Silver)

In [0]:
path = f's3://ruturaj-serverless/bank_project/silver/transactions/{yesterday}/'

transactions.write\
            .mode('overwrite')\
            .format('delta')\
            .save(path, header=True)

In [0]:
# status_values = [r[0] for r in transactions.select('status').distinct().collect()]
# payment_type_values = [r[0] for r in transactions.select('payment_type').distinct().collect()]
# currency_values = [r[0] for r in transactions.select('currency').distinct().collect()]
# location_values = [r[0] for r in transactions.select('location').distinct().collect()]

# print(status_values)
# print(payment_type_values)
# print(currency_values)
# print(location_values)

transactions_null_check = check_null_count(transactions)
transactions_null_check.display()